# DTL sanity check
Using synthetic dataset

# 1. Load synthetic dataset

In [1]:
import pandas as pd
import numpy as np
import json

# use the modular reference dataset
df_train_csv = '../data/modular_mapping_approach/syntheticdataset_test/synthetic_monte_carlo_v3_balanced.csv'
df_train = pd.read_csv(df_train_csv)


# 2. Define ruleset

In [2]:
VERSION = 'v10_claude'

ruleset_csv = f'../data/modular_mapping_approach/ruleset_epistem_default_{VERSION}.csv'
ruleset = pd.read_csv(ruleset_csv)
ruleset = ruleset.sort_values(by='priority', ascending=True)
print(ruleset)

# copy to clipboard for easy pasting into the ruleset CSV file
pd.DataFrame.to_clipboard(ruleset)

CLASS_NAMES = dict(zip(ruleset['class_id'], ruleset['class_name']))
CLASS_NAMES[0] = 'unclassified'
CLASS_NAMES[-1] = 'abstain'

# HIERARCHICAL RULESET
HIERARCHY_ID_COLS = ['id_lv_1', 'id_lv_2', 'id_lv_3', 'id_lv_4']
HIERARCHY_NAME_COLS = ['Level 1', 'Level 2', 'Level 3', 'Level 4']
HIERARCHY_COLS = [c for pair in zip(HIERARCHY_ID_COLS, HIERARCHY_NAME_COLS) for c in pair]

# columns of the ruleset that are not primitives (shared by labelling and diagnosis)
RULE_METADATA = {'class_id', 'class_name', 'priority', *HIERARCHY_COLS}

# Report ruleset primitives that are absent from the pre-extracted reference data.
# They are currently skipped by check_rule_match; review this list before accepting labels.
RULE_PRIMITIVES = [c for c in ruleset.columns if c not in RULE_METADATA]
MISSING_RULE_PRIMITIVES = [c for c in RULE_PRIMITIVES if c not in df_train.columns]
if MISSING_RULE_PRIMITIVES:
    print("WARNING: ruleset primitives absent from modular reference data:",
          MISSING_RULE_PRIMITIVES)

missing_hier = [c for c in HIERARCHY_COLS if c not in ruleset.columns]
if missing_hier:
    raise ValueError(f'Ruleset {ruleset_csv} lacks the hierarchy columns {missing_hier}')
ruleset[HIERARCHY_ID_COLS] = ruleset[HIERARCHY_ID_COLS].astype('Int64')  # keeps ids as integers despite blanks

    class_id                 class_name  id_lv_1         Level 1  id_lv_2  \
0         20                  Fish Pond       12  Non-Vegetation       25   
1         23                  Waterbody       12  Non-Vegetation       25   
2         24                 Settlement       12  Non-Vegetation       24   
3         21               Cleared Land       12  Non-Vegetation       24   
4         22                Mining area       12  Non-Vegetation       24   
5          9       Oil palm monoculture       11      Vegetation       23   
6         11        Coconut monoculture       11      Vegetation       23   
7         10          Cacao monoculture       11      Vegetation       22   
8          8         Rubber monoculture       11      Vegetation       22   
9         12          Other monoculture       11      Vegetation       22   
10         7          Plantation forest       11      Vegetation       22   
11        17                Paddy field       11      Vegetation       23   

## Hierarchical lookup
class_name -> name at every level. Blank deeper levels inherit the level above
(a class that stops at Level 3 is its own Level 4), so every class has a label at every level.
Keys are lowercase so ground-truth labels with different capitalisation still map.

In [3]:


_filled = ruleset[HIERARCHY_NAME_COLS].ffill(axis=1)
_filled.index = ruleset['class_name'].str.strip().str.casefold()

LEVEL_LOOKUP = {lvl: _filled[lvl].to_dict() for lvl in HIERARCHY_NAME_COLS}
for _d in LEVEL_LOOKUP.values():
    _d['unclassified'] = 'unclassified'
    _d['abstain'] = 'abstain'

# class order per level, following the ruleset (used to order confusion matrices)
LEVEL_ORDER = {lvl: list(dict.fromkeys(_filled[lvl])) for lvl in HIERARCHY_NAME_COLS}


def level_labels(names, level_col):
    """Map a Series of class names to their name at `level_col`; NaN where the class is not in the ruleset."""
    key = names.astype(str).str.strip().str.casefold()
    return key.map(LEVEL_LOOKUP[level_col]).where(names.notna())


print(ruleset[['class_id', 'class_name'] + HIERARCHY_NAME_COLS].to_string(index=False))

 class_id                class_name        Level 1               Level 2                   Level 3              Level 4
       20                 Fish Pond Non-Vegetation  Water Non-Vegetation                 Fish pond                  NaN
       23                 Waterbody Non-Vegetation  Water Non-Vegetation                 Waterbody                  NaN
       24                Settlement Non-Vegetation   Land Non-Vegetation                Settlement                  NaN
       21              Cleared Land Non-Vegetation   Land Non-Vegetation              Cleared Land                  NaN
       22               Mining area Non-Vegetation   Land Non-Vegetation               Mining area                  NaN
        9      Oil palm monoculture     Vegetation Non-Tree Based System            Estate/Orchard Oil palm Monoculture
       11       Coconut monoculture     Vegetation Non-Tree Based System            Estate/Orchard  Coconut monoculture
       10         Cacao monoculture     

# 3. Define and apply core DTL functions

In [4]:
def safe_num(val, default=0):
    """Convert value to float, return default if None or NaN."""
    if val is None:
        return default
    if isinstance(val, (int, float)):
        if np.isnan(val):
            return default #handling empty cells in the CSV file and turn them into the set default value (0)
        return float(val)
    try:
        return float(val)
    except:
        return default

def evaluate_condition(row_val, condition_str):
    """
    Evaluate a single condition: row_val op threshold?
    Supports OR logic with pipe separator: ">0.40|<0.10"
    
    Args:
        row_val: The value from the sample
        condition_str: Numeric expression or categorical value
    
    Returns:
        bool: True if condition is satisfied, False otherwise
    """
    if condition_str is None:
        return True  # No condition → always passes
    
    condition_str = str(condition_str).strip()
    
    # Handle OR logic (pipe-separated conditions)
    if '|' in condition_str:
        sub_conditions = [c.strip() for c in condition_str.split('|')]
        return any(evaluate_condition(row_val, c) for c in sub_conditions)

    # Handle AND logic
    if '&' in condition_str:
        sub_conditions = [c.strip() for c in condition_str.split('&')]
        return all(evaluate_condition(row_val, c) for c in sub_conditions)
    
    # Parse operator and threshold
    if condition_str.startswith('=='):
        op, threshold_str = '==', condition_str[2:]
    elif condition_str.startswith('>='):
        op, threshold_str = '>=', condition_str[2:]
    elif condition_str.startswith('<='):
        op, threshold_str = '<=', condition_str[2:]
    elif condition_str.startswith('>'):
        op, threshold_str = '>', condition_str[1:]
    elif condition_str.startswith('<'):
        op, threshold_str = '<', condition_str[1:]
    else:
        return str(row_val).strip() == condition_str
    
    threshold = safe_num(threshold_str)
    row_val_num = safe_num(row_val)
    
    if op == '>':
        return row_val_num > threshold
    elif op == '>=':
        return row_val_num >= threshold
    elif op == '<':
        return row_val_num < threshold
    elif op == '<=':
        return row_val_num <= threshold
    elif op == '==':
        return np.isclose(
            row_val_num,
            threshold,
            atol=1e-6
        )
    
    return False

def check_rule_match(row, rule):
    """
    Check if a sample row matches all conditions in a rule.
    
    Args:
        row: pd.Series with sample data
        rule: pd.Series with rule conditions
    
    Returns:
        bool: True if ALL conditions are satisfied
    """
    # Get all primitive columns (skip metadata like class_id, class_name, priority)
    metadata = {'class_id', 'class_name', 'priority'}
    primitive_cols = [col for col in rule.index if col not in metadata]
    
    for prim in primitive_cols:
        
        if prim not in row.index: 
            continue # If the primitive exists in the ruleset but does NOT exist in the modular ref data, ignore this primitive
            
        condition = rule[prim]
        
        # Handle special case for range checks
        if pd.isna(condition) or condition is None:
            continue  # No condition on this primitive
        
        # row_val = row.get(prim, np.nan)
        row_val = row[prim]
        
        if not evaluate_condition(row_val, condition):
            return False  # Any condition fails then rule doesn't match
    
    return True  # All conditions passed

def assign_label(row, ruleset):
    """
    Evaluate rules in priority order.

    - First matching rule returns its class_id.
    - Non-matching rules abstain (-1) and evaluation continues.
    - If every rule abstains, return 0 (unclassified).

    """
    ruleset_sorted = ruleset.sort_values("priority").reset_index(drop=True)

    for _, rule in ruleset_sorted.iterrows():
         if check_rule_match(row, rule):
            return rule["class_id"]  # first vote wins

    return 0  # all LFs abstained

print('Helper functions defined')

Helper functions defined


## Synthetic data quality control
Remove impossible combinations
no columns ending with _dominance overlap
if waterbody_dominance 0, then aquaculture_presence = 0 and vegetation_height = 0

In [5]:
# Keep rows with at most one active dominance category.
dominance_cols = [col for col in df_train.columns if col.endswith('_dominance')]
no_dominance_overlap = df_train[dominance_cols].sum(axis=1) <= 1

# Waterbody rows are the only ones that may have aquaculture or vegetation height.
waterbody_consistent = (
    df_train['waterbody_dominance'].ne(0)
    | (
        df_train['aquaculture_presence'].eq(0)
        # & df_train['vegetation_height'].eq(0)
    )
)

rows_before_qc = len(df_train)
df_train = df_train.loc[no_dominance_overlap & waterbody_consistent].copy()
print(f'Quality control removed {rows_before_qc - len(df_train)} rows; {len(df_train)} remain.')



Quality control removed 55136 rows; 64864 remain.


# 4. Apply DTL

In [6]:
df_train = df_train.sample(n=7000, random_state=42).copy()

df_train['label'] = df_train.apply(lambda row: assign_label(row, ruleset), axis=1)

# add class_name column
class_mapping = ruleset.set_index('class_id')['class_name'].to_dict()

df_train['class_name'] = df_train['label'].map(class_mapping).fillna('unclassified')

print(f' Labels assigned to {len(df_train)} samples\n')

print('Label distribution (assigned):')
print(
    df_train['class_name']
    .value_counts()
    .reindex(ruleset['class_name'].unique(), fill_value=0)
)

print(f'\nUnclassified (label=0): {(df_train["label"] == 0).sum()}')

 Labels assigned to 7000 samples

Label distribution (assigned):
class_name
Fish Pond                     529
Waterbody                     535
Settlement                    264
Cleared Land                  295
Mining area                   271
Oil palm monoculture           41
Coconut monoculture            37
Cacao monoculture               0
Rubber monoculture             79
Other monoculture              32
Plantation forest              11
Paddy field                    65
Other Cropland                 44
Grass or Savanna              195
Shrub                          15
Mixed/home garden            2370
Primary Dryland Forest         13
Primary Mangrove Forest         8
Primary Swamp Forest            3
Secondary Dryland Forest      110
Secondary Mangrove Forest     176
Secondary Swamp Forest        170
Name: count, dtype: int64

Unclassified (label=0): 1737


In [7]:
humanreview = pd.concat(
    [
        class_rows.sample(n=min(3, len(class_rows)), random_state=42)
        for _, class_rows in df_train.groupby('class_name', sort=False)
    ],
    ignore_index=True,
)

print(f'Selected {len(humanreview)} rows for human review across {humanreview["class_name"].nunique()} classes.')

Selected 66 rows for human review across 22 classes.


In [8]:
import os

OUTPUT_DIR = os.path.dirname(df_train_csv) or '.'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_train.to_csv(
    os.path.join(OUTPUT_DIR, f'DTL_labelled_synthetic_{VERSION}.csv'),
    index=False
)

humanreview.to_excel(
    os.path.join(OUTPUT_DIR, 'DTL_humanreview.xlsx'),
    index=False
)
